# 04 · Model Training & Experiment Tracking

Trains a baseline (class-weighted logistic regression) and a gradient-boosted model
(XGBoost) over a small hyperparameter sweep, tracking every run's params/metrics/artifacts
in **MLflow** (SQLite-backed, so the Model Registry is fully usable locally). The best run
is registered as `fraud-xgboost` and promoted to the `staging` alias — the same registry
this project's real-time API and retraining pipeline (later phases) read from.

**Why class weighting instead of SMOTE/oversampling:** several features here are
time-windowed velocity aggregates (transactions in the trailing 1h/24h/7d). Interpolating
synthetic minority-class rows (as SMOTE does) would fabricate physically incoherent
combinations of those aggregates. `scale_pos_weight` reweights the loss instead of
touching the data, which is the safer default for temporally-structured fraud features.

**Why PR-AUC (average precision) as the primary model-selection metric:** with ~1.5-2%
positive rate, ROC-AUC is dominated by the (easy) true-negative volume and can look
deceptively strong. PR-AUC is far more sensitive to the trade-off that actually matters
operationally — precision at usable recall.

In [1]:
import os, json
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

import mlflow
import mlflow.sklearn
import mlflow.xgboost
from mlflow.tracking import MlflowClient

with open("../data/processed/feature_columns.json") as f:
    cfg = json.load(f)
FEATURES, LABEL = cfg["features"], cfg["label"]

train = pd.read_parquet("../data/processed/train.parquet")
val = pd.read_parquet("../data/processed/val.parquet")
test = pd.read_parquet("../data/processed/test.parquet")

X_train, y_train = train[FEATURES], train[LABEL]
X_val, y_val = val[FEATURES], val[LABEL]
X_test, y_test = test[FEATURES], test[LABEL]

print(f"train={X_train.shape}  val={X_val.shape}  test={X_test.shape}")
print(f"train fraud rate={y_train.mean():.4%}  val={y_val.mean():.4%}  test={y_test.mean():.4%}")

train=(734002, 19)  val=(157287, 19)  test=(157286, 19)
train fraud rate=0.5925%  val=0.4476%  test=0.6059%


In [2]:
os.makedirs("../models", exist_ok=True)
mlflow.set_tracking_uri(f"sqlite:///{os.path.abspath('../mlflow.db')}")
mlflow.set_experiment("fraud-detection")
print("MLflow tracking URI:", mlflow.get_tracking_uri())

2026/08/25 19:43:46 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/08/25 19:43:46 INFO mlflow.store.db.utils: Updating database tables


2026/08/25 19:43:47 INFO mlflow.tracking.fluent: Experiment with name 'fraud-detection' does not exist. Creating a new experiment.


MLflow tracking URI: sqlite:////Users/crysis/fraud_detection/mlflow.db


## Baseline: class-weighted logistic regression

A simple, well-calibrated linear baseline. Any tree ensemble we ship needs to clearly beat this to justify its extra complexity and reduced interpretability.

In [3]:
with mlflow.start_run(run_name="logreg_baseline") as run:
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s = scaler.transform(X_val)

    logreg = LogisticRegression(class_weight="balanced", max_iter=2000, C=1.0, random_state=42)
    logreg.fit(X_train_s, y_train)

    logreg_val_proba = logreg.predict_proba(X_val_s)[:, 1]
    logreg_val_roc_auc = roc_auc_score(y_val, logreg_val_proba)
    logreg_val_pr_auc = average_precision_score(y_val, logreg_val_proba)

    mlflow.log_params({"model": "logreg", "C": 1.0, "class_weight": "balanced"})
    mlflow.log_metrics({"val_roc_auc": logreg_val_roc_auc, "val_pr_auc": logreg_val_pr_auc})
    mlflow.sklearn.log_model(logreg, "model")

    logreg_run_id = run.info.run_id
    print(f"LogReg baseline  val ROC-AUC={logreg_val_roc_auc:.4f}  val PR-AUC={logreg_val_pr_auc:.4f}")

2026/08/25 19:43:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


LogReg baseline  val ROC-AUC=0.9386  val PR-AUC=0.2116


## XGBoost hyperparameter sweep

A small, deliberately narrow grid (this is a portfolio project, not a Kaggle leaderboard chase) — each configuration is logged as its own MLflow run for full comparability.

In [4]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight = {scale_pos_weight:.1f}")

param_grid = [
    dict(max_depth=4, learning_rate=0.10, n_estimators=200),
    dict(max_depth=6, learning_rate=0.10, n_estimators=300),
    dict(max_depth=6, learning_rate=0.05, n_estimators=500),
    dict(max_depth=8, learning_rate=0.05, n_estimators=400),
]

sweep_results = []
sweep_models = {}

for params in param_grid:
    run_name = f"xgb_d{params['max_depth']}_lr{params['learning_rate']}_n{params['n_estimators']}"
    with mlflow.start_run(run_name=run_name) as run:
        model = xgb.XGBClassifier(
            **params,
            scale_pos_weight=scale_pos_weight,
            objective="binary:logistic",
            eval_metric="aucpr",
            random_state=42,
            n_jobs=-1,
            tree_method="hist",
        )
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

        val_proba = model.predict_proba(X_val)[:, 1]
        roc = roc_auc_score(y_val, val_proba)
        pr = average_precision_score(y_val, val_proba)

        mlflow.log_params({**params, "scale_pos_weight": round(scale_pos_weight, 2), "model": "xgboost"})
        mlflow.log_metrics({"val_roc_auc": roc, "val_pr_auc": pr})
        mlflow.xgboost.log_model(model, "model")

        sweep_results.append({"run_id": run.info.run_id, "val_roc_auc": roc, "val_pr_auc": pr, **params})
        sweep_models[run.info.run_id] = model
        print(f"{run_name:35s}  val ROC-AUC={roc:.4f}  val PR-AUC={pr:.4f}")

results_df = pd.DataFrame(sweep_results).sort_values("val_pr_auc", ascending=False).reset_index(drop=True)
results_df

scale_pos_weight = 167.8


2026/08/25 19:43:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


xgb_d4_lr0.1_n200                    val ROC-AUC=0.9913  val PR-AUC=0.6715


2026/08/25 19:44:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


xgb_d6_lr0.1_n300                    val ROC-AUC=0.9900  val PR-AUC=0.6716


2026/08/25 19:44:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


xgb_d6_lr0.05_n500                   val ROC-AUC=0.9911  val PR-AUC=0.6754


2026/08/25 19:44:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


xgb_d8_lr0.05_n400                   val ROC-AUC=0.9912  val PR-AUC=0.7002


,run_id,val_roc_auc,val_pr_auc,max_depth,learning_rate,n_estimators
0,6e31673d849a40cfbd127323b186e17a,0.991171,0.700216,8,0.05,400
1,d9d7817f24c74141969548267326f092,0.991099,0.675412,6,0.05,500
2,5a0ad09835a147ffb3213294effb8431,0.989976,0.671573,6,0.10,300
3,df5474b0ad7e4c6abe5325de0aab8432,0.991291,0.671492,4,0.10,200


## Select and register the best model

Promoted to the `staging` alias in the MLflow Model Registry — this is the handoff point where `05_model_evaluation_explainability.ipynb` picks the model up for a rigorous hold-out evaluation before anything is promoted further.

In [5]:
best = results_df.iloc[0]
best_run_id = best.run_id
best_model = sweep_models[best_run_id]

print(f"Best config: depth={int(best.max_depth)} lr={best.learning_rate} n_estimators={int(best.n_estimators)}")
print(f"XGBoost val PR-AUC={best.val_pr_auc:.4f}  vs  LogReg baseline val PR-AUC={logreg_val_pr_auc:.4f}")

model_uri = f"runs:/{best_run_id}/model"
registered = mlflow.register_model(model_uri, "fraud-xgboost")

client = MlflowClient()
client.set_registered_model_alias("fraud-xgboost", "staging", registered.version)
print(f"Registered 'fraud-xgboost' version {registered.version} -> alias 'staging'")

Successfully registered model 'fraud-xgboost'.
2026/08/25 19:44:38 WARNING mlflow.tracking._model_registry.fluent: Run with id 6e31673d849a40cfbd127323b186e17a has no artifacts at artifact path 'model', registering model based on models:/m-60969db5532f4afe8eb9e7eef27af437 instead


Best config: depth=8 lr=0.05 n_estimators=400
XGBoost val PR-AUC=0.7002  vs  LogReg baseline val PR-AUC=0.2116
Registered 'fraud-xgboost' version 1 -> alias 'staging'


Created version '1' of model 'fraud-xgboost'.


## Persist a portable model artifact

Saved outside MLflow too (`models/`) so the FastAPI scoring service can load it directly without depending on a running MLflow tracking server in production.

In [6]:
best_model.save_model("../models/fraud_xgboost.json")
with open("../models/feature_columns.json", "w") as f:
    json.dump({"features": FEATURES, "label": LABEL, "mlflow_run_id": best_run_id,
               "mlflow_model_version": registered.version}, f, indent=2)

print("Saved models/fraud_xgboost.json + models/feature_columns.json")
print("\nRun `mlflow ui --backend-store-uri sqlite:///mlflow.db` from the project root to browse experiments.")

Saved models/fraud_xgboost.json + models/feature_columns.json

Run `mlflow ui --backend-store-uri sqlite:///mlflow.db` from the project root to browse experiments.
